# Hybrid Fraud Prioritisation (Final Decision Logic)

This notebook implements the **final decision logic** for fraud detection by combining:

- a **supervised transaction-level fraud score**, and  
- an **unsupervised account-day behavioural anomaly score**.

No new models are trained in this notebook.  

Instead, the goal is to **prioritise transactions for investigation** by combining transaction risk with behavioural context.

The hybrid approach improves fraud concentration at the top of the investigation list,
making it more effective for real-world fraud analysis where human review capacity is limited.


## 1.Setup Environment and Load Model Outputs

In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#load required libraries

import pandas as pd
import numpy as np
import os

from datasets import load_dataset
from sklearn.preprocessing import MinMaxScaler

import joblib

In [ ]:
#load saved results from supervised and unsupervised models
output_dir='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/outputs'

supervised_scores=pd.read_parquet(f"{output_dir}/supervised_scores.parquet")
anomaly_results = pd.read_parquet(f"{output_dir}/anomaly_results.parquet")

print(supervised_scores.shape)
print(anomaly_results.shape)

## 2.Merge Transaction Risk with Behavioural Context

In [ ]:
#merge behavioural anomaly scores to transaction-level supervised scores

hybrid_df=supervised_scores.merge(
    anomaly_results, on=['nameOrig', 'day'], how='left'
)

In [ ]:
hybrid_df.shape

In [ ]:
#inspect the structure of the merged dataframe
hybrid_df.info()

In [ ]:
#check for the missing values in the merged dataframe
hybrid_df.isnull().sum()


## 3.Normalise Anomaly Scores

In [ ]:
#normalise the anomlay scores for the combination
scaler = MinMaxScaler()
hybrid_df['anomaly_score_norm']=scaler.fit_transform(hybrid_df[['anomaly_score']])

## 4.Create Hybrid Fraud Score

In [ ]:
#compute a weighted hybrid fraud risk score, keeping the supervised prediction dominant while using anomaly signals as contextual adjustment
hybrid_df['hybrid_score']=(
    0.7*hybrid_df['supervised_score']+0.3*hybrid_df['anomaly_score_norm']
)

In [ ]:
hybrid_df.info()

## 5.Rank and Evaluate Prioritisation Performance

In [ ]:
#evaluate how much fraud is concentrated at different top-K cut-offs compared to the overall baseline rate

#compute baseline fraud rate across the dataframe
baseline=hybrid_df['isFraud'].mean()
print('Baseline fraud rate:', baseline)

#rank transactions by predicted fraud risk to compare supervised-only and hybrid scoring
supervised_ranked=hybrid_df.sort_values('supervised_score', ascending=False)
hybrid_ranked=hybrid_df.sort_values('hybrid_score', ascending=False)

#define top-k thresholds to assess early fraud concentration at multiple percentage-based cutoffs
n=len(hybrid_df)
ks=[100, 500, int(0.001*n), int(0.005*n), int(0.01*n)]

#compute fraud-rate enrichment at the top of the ranking for the supervised model
print('\nSupervised:')
for k in ks:
  fr=supervised_ranked.head(k)['isFraud'].mean()
  enrichment=fr/baseline
  print(f'Top {k:>5} | Fraud_rate: {fr:.4f} | Enrichment: {enrichment:.1f}x')

#compute fraud-rate enrichment for the hybrid ranking to quantify added behavioural signal value
print('\nHybrid:')
for k in ks:
  fr=hybrid_ranked.head(k)['isFraud'].mean()
  enrichment=fr/baseline
  print(f'Top {k:>5} | Fraud_rate: {fr:.4f} | Enrichment: {enrichment:.1f}x')

**Notes:**

- The supervised LightGBM model shows moderate fraud prioritisation, improving the baseline fraud rate by up to six times within the top-ranked transactions, with its strongest performance around the Top-1000 cutoff.


- When behavioural anomaly scores were added in a hybrid ranking, performance improves significantly at the very top of the list. In particular, the fraud rate in the top 100 transactions increased from 2% to 7%, representing a 3.5x improvement over the supervised model alone. While the hybrid advantage diminishes at larger cutoffs, this is expected, as the hybrid ranking concentrates a higher proportion of fraud cases at the very top of the list, leaving fewer fraud instances in later ranks.


- This result highlights that behavioural context provides complementary information that enhances the identification of the most critical fraud cases.